# Module 2 â€” Problem Framing & Config
**Type:** [Code With Me]  
**Time:** 20 minutes  
**Job:** Lock every constant the workshop depends on. Define pooled scoring mechanics. Connect the evaluation standard to an inventory decision.

---

> **Instructor note:** This module has exactly two [Code With Me] cells. Everything else is [Watch Only]. Students fill in the CV parameter block and the scoring function call â€” both are 2 lines. The goal is to get everyone touching `config.py` imports and the evaluation API before we hit data.

---
## 2.1 â€” Path Setup
**[Watch Only]**

> **Instructor note (1 min):** Run this cell silently. Explain that this is the only cell in any notebook that touches `sys.path` â€” everything else imports cleanly because this ran first.

In [ ]:
import sys
from pathlib import Path

# Add project root to path â€” required for config.py and src/ imports
PROJECT_ROOT = Path().resolve().parent  # notebooks live one level below root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

---
## 2.2 â€” Import Config
**[Watch Only]**

> **Instructor note (2 min):** Import the config and print the key values. Make the point explicitly: every number you see here is the single authoritative version. If a student redefines `HORIZON = 14` in their own cell later, they are off the workshop path.

In [ ]:
import pickle
import pandas as pd
import numpy as np

import config
from config import (
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    RANDOM_SEED,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    INTERVAL_COVERAGE,
    ARTIFACT_DIR,
    WORKSHOP_SUBSET_PATH,
)

print("Workshop configuration")
print("=" * 40)
print(f"  Forecast horizon    : {HORIZON} days")
print(f"  Season length       : {SEASON_LENGTH} days (weekly)")
print(f"  CV windows          : {N_WINDOWS}")
print(f"  Step size           : {STEP_SIZE} days")
print(f"  Refit between windows: {REFIT}")
print(f"  Random seed         : {RANDOM_SEED}")
print(f"  Live subset (micro) : {MICRO_SUBSET_N} series")
print(f"  Full subset         : {WORKSHOP_SUBSET_N} series")
print(f"  Interval coverage   : {int(INTERVAL_COVERAGE * 100)}%")

---
## 2.3 â€” Why These Parameters?
**[Watch Only]**

> **Instructor note (3 min):** Walk through each parameter group with the business framing below. Do not just read the table â€” connect each number to an operational consequence. The REFIT=False decision always generates questions; address it before someone asks.

These are not arbitrary. Each parameter maps to a production constraint.

**Horizon = 28 days**  
Standard replenishment cycle for Walmart-scale retail. Your procurement team orders 4 weeks out. A 7-day horizon is operationally useless for inventory planning at that lead time.

**Season length = 7 days**  
Weekly demand cycles dominate retail. Friday â‰  Monday. Any model that does not capture this will have structurally biased errors.

**N_WINDOWS = 3, STEP_SIZE = 28**  
Three non-overlapping 28-day evaluation windows. This gives us 84 days of out-of-sample signal across 1,000 series â€” enough to be statistically meaningful, not so much that it consumes the training history.

**REFIT = False**  
We train once and roll the evaluation window forward. In production, daily retraining is expensive. Testing with `REFIT=True` would inflate model performance beyond what you'd see in a real deployment.

**MICRO_SUBSET_N = 50**  
Live cells run on 50 series to stay inside the 90-second ceiling. Precomputed artifacts cover all 1,000. The patterns are identical â€” scale is the only difference.

---
## 2.4 â€” Define the CV Parameters
**[Code With Me â€” 2 lines]**

> **Instructor note (3 min):** Students fill in `n_windows` and `step_size` from config imports. Pause and wait for everyone to complete it. This is the first active learning moment â€” keep it simple and successful. The point is muscle memory on "import from config, do not hardcode."

In [ ]:
# Cross-validation parameters for StatsForecast.
# Import the values from config â€” do not hardcode them here.

cv_params = {
    "h":         HORIZON,
    "n_windows": N_WINDOWS,    # __FILL_IN__: import from config
    "step_size": STEP_SIZE,    # __FILL_IN__: import from config
    "refit":     REFIT,
}

print("CV parameters locked:")
for k, v in cv_params.items():
    print(f"  {k:<12}: {v}")

**Expected output:**
```
CV parameters locked:
  h           : 28
  n_windows   : 3
  step_size   : 28
  refit       : False
```

---
## 2.5 â€” Evaluation Logic: wMAPE
**[Watch Only]**

> **Instructor note (3 min):** Show the formula, then immediately connect it to procurement. The key insight is pooling â€” say it explicitly. "If you average per series first, a SKU selling 1 unit/day has the same weight as one selling 10,000. That is not how inventory risk works."

**wMAPE â€” Weighted Mean Absolute Percentage Error**

$$\text{wMAPE} = \frac{\sum |y_t - \hat{y}_t|}{\sum |y_t|}$$

Pooled across every observation, every series, every CV window. The denominator sums actual sales volume â€” so high-volume SKUs dominate the score. That is intentional. A 20% error on a 10,000-unit SKU costs more than a 20% error on a 1-unit SKU.

**The inventory risk connection:**  
Every percentage point of wMAPE maps to safety stock cost. If your procurement team holds 2 weeks of safety stock to buffer forecast error, cutting wMAPE by 5 points is not a modeling achievement â€” it is a working capital decision.

---
## 2.6 â€” Evaluation Logic: Interval Score
**[Watch Only]**

> **Instructor note (2 min):** The formula looks intimidating. Skip deriving it â€” go straight to the intuition. "Wide intervals that miss are worse than narrow intervals that hit. Both dimensions are penalized."

**Interval Score (Winkler Score) at 80%**

$$\text{IS}_{\alpha} = (\hat{u} - \hat{l}) + \frac{2}{\alpha}(\hat{l} - y)^+ + \frac{2}{\alpha}(y - \hat{u})^+$$

Where $\hat{l}$, $\hat{u}$ are the lower and upper bounds of the 80% interval, $\alpha = 0.20$, and $(x)^+ = \max(x, 0)$.

**What this penalizes:**
- **Width:** A wide interval is penalized even if every actual falls inside it
- **Misses:** If the actual falls outside the interval, a steep penalty kicks in (scaled by $2/\alpha = 10$)

**Why point forecasts alone are dangerous:**  
If your model predicts 100 units but the 80% interval runs from 10 to 500, that forecast cannot drive a purchasing decision. Interval Score surfaces this problem. A model that wins on wMAPE but loses badly on Interval Score is not production-ready.

---
## 2.7 â€” Verify the Scoring Functions
**[Code With Me â€” 2 lines]**

> **Instructor note (3 min):** Students call `pooled_wmape` and `pooled_interval_score` on the toy arrays. The fill-in is the function arguments â€” just `y_true` and `y_hat`, and `y_true`, `lo`, `hi`. This validates their `src/` import path is working before we load any real data.

In [ ]:
from src.evaluation import pooled_wmape, pooled_interval_score

# Toy arrays â€” verify the functions return sensible numbers
y_true = np.array([100.0, 200.0, 150.0, 80.0, 120.0])
y_hat  = np.array([110.0, 180.0, 160.0, 70.0, 130.0])
lo     = np.array([ 80.0, 150.0, 120.0, 50.0,  90.0])
hi     = np.array([140.0, 220.0, 200.0, 110.0, 160.0])

wmape = pooled_wmape(y_true, y_hat)          # __FILL_IN__: call pooled_wmape
iscore = pooled_interval_score(y_true, lo, hi)  # __FILL_IN__: call pooled_interval_score

print(f"wMAPE         : {wmape:.4f}  ({wmape*100:.2f}%)")
print(f"Interval Score: {iscore:.4f}")
print()
print("Both functions are working. Importing src.evaluation is confirmed.")

**Expected output:**
```
wMAPE         : 0.0923  (9.23%)
Interval Score: 60.0000

Both functions are working. Importing src.evaluation is confirmed.
```

> **Instructor note:** If a student gets an `ImportError` here, their `sys.path` is not set correctly. Point them to Cell 2.1 â€” re-run the path setup cell, then retry.

---
## 2.8 â€” Save the Config Artifact
**[Watch Only]**

> **Instructor note (1 min):** Run this quickly. The artifact is a serialized dict of the key config values â€” used by `00_env_check.ipynb` to confirm config was locked before the session. Mention that the naming convention (`02_` prefix) is the artifact registry rule.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

global_config = {
    "HORIZON":           HORIZON,
    "SEASON_LENGTH":     SEASON_LENGTH,
    "N_WINDOWS":         N_WINDOWS,
    "STEP_SIZE":         STEP_SIZE,
    "REFIT":             REFIT,
    "RANDOM_SEED":       RANDOM_SEED,
    "MICRO_SUBSET_N":    MICRO_SUBSET_N,
    "WORKSHOP_SUBSET_N": WORKSHOP_SUBSET_N,
    "INTERVAL_COVERAGE": INTERVAL_COVERAGE,
    "cv_params":         cv_params,
}

config_path = ARTIFACT_DIR / "02_global_config.pkl"
with open(config_path, "wb") as f:
    pickle.dump(global_config, f)

print(f"  âœ“ Config artifact saved: {config_path.name}")

**Expected output:**
```
  âœ“ Config artifact saved: 02_global_config.pkl
```

---
## 2.9 â€” The Enterprise Cliff
**[Watch Only]**

> **Instructor note (1 min):** Read this section or paraphrase it. This is the first "earned incompleteness" moment â€” explicitly naming what we are simplifying and why. It sets the tone for the authority hooks in Modules 5 and 6.

Everything in this module is clean and fast because we are working with a locked, pre-filtered subset.

In a real enterprise deployment, this step looks different:

- **Horizon is not fixed.** Different SKU categories have different lead times. A global `HORIZON = 28` is a simplification. Production systems often run multiple horizon configurations simultaneously.
- **CV windows are not uniform.** Promotional periods, stockouts, and data anomalies require custom exclusion windows. Automated CV on raw data will quietly evaluate against corrupted actuals.
- **The scoring function is not the business metric.** wMAPE is a proxy. The actual KPI is safety stock reduction, stockout rate, or gross margin impact. Connecting model evaluation to financial outcomes requires a separate translation layer.

For this workshop, the locked config is correct and sufficient. But the gap between this cell and a production config management system is where the real engineering lives.

---

> **Instructor note â€” transition:** "Config is locked. Scoring functions are verified. Let's go look at the data."
>
> Open `instructor_notebooks/03_eda_and_health.ipynb`.